# 08 · Registro en Databricks / MLflow (Serverless)

Registra el modelo congelado `models/model.joblib` en **MLflow** y en el **Model Registry** (Unity Catalog).

| Qué | Nombre |
|---|---|
| Experimento y corrida | `AccessGuard_ZeroTrust_Experiment` |
| Modelo registrado | `AccessGuard_ZeroTrust_Model` (en Unity Catalog: `<catálogo>.<esquema>.AccessGuard_ZeroTrust_Model`) |

**Cómo usarlo en Databricks:** clona el repositorio como *Git folder*, abre este notebook sobre cómputo **Serverless** y ejecuta las celdas en orden. El mismo notebook funciona en local (usa una base SQLite de respaldo) para poder probarlo antes.

> Las métricas se **leen** de `reports/07_shap_analysis.json` (no se recalculan ni se usa la reserva para decidir nada). Las cifras de la reserva se registran como `holdout_*` solo para trazabilidad.

### 1. Dependencias (solo Databricks)
`accessguard.modeling` importa XGBoost, LightGBM y CatBoost al cargar el paquete; en Serverless hay que instalarlos. Después de esta celda el Python se reinicia.

In [ ]:
# %pip install -q catboost lightgbm xgboost scipy
# dbutils.library.restartPython()
# (descomenta las dos líneas anteriores en Databricks; en local ya están instaladas)

In [ ]:
import os, sys, json
from pathlib import Path
import numpy as np, pandas as pd, joblib, mlflow, sklearn
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient

ROOT = Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT          # funciona dentro de un Git folder de Databricks
sys.path.insert(0, str(ROOT / "src")); sys.path.insert(0, str(ROOT))
from accessguard.serving import AccessGuardZeroTrust

IN_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ
MODEL_NAME = "AccessGuard_ZeroTrust_Model"
EXPERIMENT_NAME = "AccessGuard_ZeroTrust_Experiment"
RUN_NAME = "AccessGuard_ZeroTrust_Experiment"
CATALOG, SCHEMA = "workspace", "default"          # AJUSTA a tu catálogo/esquema de Unity Catalog

if IN_DATABRICKS:
    mlflow.set_registry_uri("databricks-uc")
    REGISTERED_NAME = f"{CATALOG}.{SCHEMA}.{MODEL_NAME}"
    user = spark.sql("SELECT current_user()").first()[0]            # noqa: F821  (spark existe en Databricks)
    mlflow.set_experiment(f"/Users/{user}/{EXPERIMENT_NAME}")
else:
    mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", f"sqlite:///{ROOT / 'mlflow_local.db'}"))
    REGISTERED_NAME = MODEL_NAME
    mlflow.set_experiment(EXPERIMENT_NAME)
print("Entorno:", "Databricks" if IN_DATABRICKS else "local", "· modelo registrado:", REGISTERED_NAME, "· MLflow", mlflow.__version__)

### 2. Cargar el artefacto congelado y las métricas del notebook 07

In [ ]:
MODEL_PATH = ROOT / "models" / "model.joblib"
bundle = joblib.load(MODEL_PATH)
rep = json.loads((ROOT / "reports" / "07_shap_analysis.json").read_text(encoding="utf-8"))
assert abs(bundle["t_auto_strict"] - rep["thresholds"]["t_auto_strict"]) < 1e-12, "model.joblib y el reporte 07 no coinciden: reejecuta el notebook 07"

zones_oof = {r["zona"]: r for r in rep["oof_zones"]}
zones_hold = {r["zona"]: r for r in rep["holdout"]["zones"]}
cap = rep["holdout"]["queue_capture"]
metrics = {
    # Zero Trust (OOF = validación cruzada por gerente: base de las decisiones)
    "oof_roc_auc": bundle["cv_metrics_oof"]["roc_auc"],
    "oof_pr_auc_deny": bundle["cv_metrics_oof"]["pr_auc_deny"],
    "oof_tasa_autoaprobacion_estricta": zones_oof["verde"]["% del total"],
    "oof_fpr_upper95": bundle["cv_metrics_oof"]["fpr_upper95"],
    # Reserva (solo trazabilidad, no se decide con ella)
    "holdout_roc_auc": rep["holdout"]["roc_auc"],
    "holdout_pr_auc_deny": rep["holdout"]["pr_auc_deny"],
    "holdout_tasa_autoaprobacion_estricta": zones_hold["verde"]["% del total"],
    "holdout_accesos_indebidos_verde": zones_hold["verde"]["accesos indebidos"],
    "holdout_fpr_upper95": rep["holdout"]["strict_fpr_upper95"],
    "holdout_captura_cola_5pct": cap["5%"], "holdout_captura_cola_10pct": cap["10%"], "holdout_captura_cola_20pct": cap["20%"],
    "holdout_pct_zona_roja": zones_hold["rojo"]["% del total"],
    "latencia_mediana_ms": bundle["latency_ms"]["single_median_ms"],
}
pd.Series(metrics).round(4)

### 3. Corrida de MLflow: parámetros, métricas, modelo y Registry

In [ ]:
params = {"modelo": bundle["model_name"], **{f"hp_{k}": v for k, v in bundle["best_params"].items()},
          "t_auto_strict": bundle["t_auto_strict"], "t_low": bundle["t_low"], "confianza_practica": bundle["t_confidence"],
          "n_variables": len(bundle["feature_names"]), "esquema_cv": bundle["cv_metrics_oof"]["scheme"],
          "marco": bundle["framework"], "sklearn": sklearn.__version__}

# ejemplo de entrada con los tipos que usa el servicio (códigos enteros); no se usa para decidir nada
X_example = pd.read_csv(ROOT / "Data" / "processed" / "holdout_clean.csv").drop(columns="ACTION").head(5).astype("int64") \
    if (ROOT / "Data" / "processed" / "holdout_clean.csv").exists() else \
    pd.DataFrame({c: [1, 2, 3] for c in bundle["input_columns"]}, dtype="int64")
wrapper = AccessGuardZeroTrust()
class _Ctx:  # contexto mínimo para probar el wrapper antes de registrarlo
    artifacts = {"bundle": str(MODEL_PATH)}
wrapper.load_context(_Ctx)
signature = infer_signature(X_example, wrapper.predict(_Ctx, X_example))

with mlflow.start_run(run_name=RUN_NAME) as run:
    mlflow.log_params(params)
    mlflow.log_metrics({k: float(v) for k, v in metrics.items()})
    mlflow.set_tags({"framework": "Zero Trust / IGA", "principio": "Menor Privilegio", "decision": "nunca auto-deniega",
                     "regla_zonas": "verde p>t_auto · amarillo triaje · rojo p<t_low"})
    mlflow.log_artifact(str(ROOT / "reports" / "07_shap_analysis.json"))
    if (ROOT / "reports" / "figures" / "shap_summary.png").exists():
        mlflow.log_artifact(str(ROOT / "reports" / "figures" / "shap_summary.png"))
    info = mlflow.pyfunc.log_model(
        name="model", python_model=AccessGuardZeroTrust(), artifacts={"bundle": str(MODEL_PATH)},
        code_paths=[str(ROOT / "src" / "accessguard")], signature=signature, input_example=X_example,
        pip_requirements=[f"scikit-learn=={sklearn.__version__}", f"pandas=={pd.__version__}", f"numpy=={np.__version__}",
                          "scipy", "joblib", "catboost", "lightgbm", "xgboost"],
        registered_model_name=REGISTERED_NAME)
    run_id = run.info.run_id
print("Corrida:", run_id, "· modelo:", info.model_uri)

### 4. Alias `champion` y comprobación
Unity Catalog usa **alias** (no *stages*). Se carga el modelo desde el Registry y se comprueba que da lo mismo que el archivo.

In [ ]:
client = MlflowClient()
versions = client.search_model_versions(f"name='{REGISTERED_NAME}'")
latest = max(int(v.version) for v in versions)
client.set_registered_model_alias(REGISTERED_NAME, "champion", latest)

loaded = mlflow.pyfunc.load_model(f"models:/{REGISTERED_NAME}@champion")
out = loaded.predict(X_example)
ref = bundle["pipeline"].predict_proba(X_example[bundle["input_columns"]])[:, 1]
assert np.allclose(out["p_approve"], ref), "el modelo registrado no coincide con model.joblib"
print(f"{REGISTERED_NAME} versión {latest} → alias 'champion' · comprobación OK")
out.round(4)

### 5. Qué queda registrado y qué no
* **Sí:** hiperparámetros, umbrales (`t_auto_strict`, `t_low`), métricas Zero Trust (OOF y reserva), figura SHAP, firma de entrada/salida y el código de `src/accessguard`.
* **No:** datos (el conjunto es anónimo, pero no se sube) ni credenciales. La app de Streamlit **no** consulta Databricks: carga `models/model.joblib`.
* Si la tasa de auto-aprobación cae o aparece un acceso indebido en la zona verde en producción, el modelo vuelve a revisión humana total (la política por defecto es "revisar todo").